# PAMAP2 Machine Learning Preparation

This notebook generates fixed-size sliding windows and extracts handcrafted statistical and frequency-domain features for traditional Machine Learning models.

In [5]:
## Import Libraries

In [6]:
import numpy as np
import pandas as pd

In [7]:
## Load Preprocessed Dataset

In [8]:
DATA_PATH = "/lustre09/project/6081099/reem2005/DATASET/pamap2_preprocessed.csv"

data = pd.read_csv(DATA_PATH)

print(data.shape)

data.head()

(1936481, 42)


,timestamp,activityID,hand_temperature,hand_acc16_x,hand_acc16_y,hand_acc16_z,hand_acc6_x,hand_acc6_y,hand_acc6_z,hand_gyro_x,...,ankle_acc6_x,ankle_acc6_y,ankle_acc6_z,ankle_gyro_x,ankle_gyro_y,ankle_gyro_z,ankle_mag_x,ankle_mag_y,ankle_mag_z,subject
0,37.66,1,30.375,2.21530,8.27915,5.58753,2.24689,8.55387,5.77143,-0.004750,...,9.63162,-1.76757,0.265761,0.002908,-0.027714,0.001752,-61.1081,-36.8636,-58.3696,101
1,37.67,1,30.375,2.29196,7.67288,5.74467,2.27373,8.14592,5.78739,-0.171710,...,9.58649,-1.75247,0.250816,0.020882,0.000945,0.006007,-60.8916,-36.3197,-58.3656,101
2,37.68,1,30.375,2.29090,7.14240,5.82342,2.26966,7.66268,5.78846,-0.238241,...,9.60196,-1.73721,0.356632,-0.035392,-0.052422,-0.004882,-60.3407,-35.7842,-58.6119,101
3,37.69,1,30.375,2.21800,7.14365,5.89930,2.22177,7.25535,5.88000,-0.192912,...,9.58674,-1.78264,0.311453,-0.032514,-0.018844,0.026950,-60.7646,-37.1028,-57.8799,101
4,37.70,1,30.375,2.30106,7.25857,6.09259,2.20720,7.24042,5.95555,-0.069961,...,9.64677,-1.75240,0.295902,0.001351,-0.048878,-0.006328,-60.2040,-37.1225,-57.8847,101


## Sliding Window Parameters

In [9]:
WINDOW_SIZE = 200
STEP_SIZE = 100

## Sensor Features

In [10]:
sensor_cols = [
    col
    for col in data.columns
    if col not in [
        "timestamp",
        "activityID",
        "subject"
    ]
]

print("Number of sensor channels:", len(sensor_cols))

Number of sensor channels: 39


## Feature Names

In [11]:
feature_names = []

for col in sensor_cols:

    feature_names.extend([
        f"{col}_mean",
        f"{col}_median",
        f"{col}_std",
        f"{col}_max",
        f"{col}_min",
        f"{col}_range",
        f"{col}_iqr",
        f"{col}_rms",
        f"{col}_energy",
        f"{col}_fft_mean",
        f"{col}_fft_std",
        f"{col}_fft_energy",
        f"{col}_dominant_freq"
    ])

print("Total Features:", len(feature_names))

Total Features: 507


In [15]:
X = []
y = []
groups = []

for subject in sorted(data["subject"].unique()):

    for activity in sorted(data["activityID"].unique()):

        subject_activity = data[
            (data["subject"] == subject) &
            (data["activityID"] == activity)
        ].reset_index(drop=True)

        if len(subject_activity) < WINDOW_SIZE:
            continue

        for start in range(
            0,
            len(subject_activity) - WINDOW_SIZE + 1,
            STEP_SIZE
        ):

            window = subject_activity.iloc[
                start:start + WINDOW_SIZE
            ]

            features = []

            for col in sensor_cols:

                signal = window[col].values

                # ==========================
                # Time-domain features
                # ==========================
                mean_val = np.mean(signal)
                median_val = np.median(signal)
                std_val = np.std(signal)
                max_val = np.max(signal)
                min_val = np.min(signal)

                range_val = max_val - min_val

                q75, q25 = np.percentile(signal, [75, 25])
                iqr_val = q75 - q25

                rms_val = np.sqrt(np.mean(signal ** 2))
                energy_val = np.sum(signal ** 2)

                # ==========================
                # Frequency-domain features
                # ==========================
                fft_signal = np.abs(np.fft.rfft(signal))

                fft_mean = np.mean(fft_signal)
                fft_std = np.std(fft_signal)
                fft_energy = np.sum(fft_signal ** 2)

                # Ignore DC component
                dominant_freq = np.argmax(fft_signal[1:]) + 1

                features.extend([
                    mean_val,
                    median_val,
                    std_val,
                    max_val,
                    min_val,
                    range_val,
                    iqr_val,
                    rms_val,
                    energy_val,
                    fft_mean,
                    fft_std,
                    fft_energy,
                    dominant_freq
                ])

            X.append(features)
            y.append(activity)
            groups.append(subject)

X = np.array(X)
y = np.array(y)
y = y - 1
groups = np.array(groups)

print("=" * 50)
print("Feature Extraction Finished")
print("=" * 50)
print("Number of windows :", len(X))
print("X shape           :", X.shape)
print("y shape           :", y.shape)
print("groups shape      :", groups.shape)

Feature Extraction Finished
Number of windows : 19226
X shape           : (19226, 507)
y shape           : (19226,)
groups shape      : (19226,)


In [16]:
features_df = pd.DataFrame(X, columns=feature_names)

features_df["Activity"] = y
features_df["Subject"] = groups

features_df.head()

,hand_temperature_mean,hand_temperature_median,hand_temperature_std,hand_temperature_max,hand_temperature_min,hand_temperature_range,hand_temperature_iqr,hand_temperature_rms,hand_temperature_energy,hand_temperature_fft_mean,...,ankle_mag_z_range,ankle_mag_z_iqr,ankle_mag_z_rms,ankle_mag_z_energy,ankle_mag_z_fft_mean,ankle_mag_z_fft_std,ankle_mag_z_fft_energy,ankle_mag_z_dominant_freq,Activity,Subject
0,30.38625,30.3750,0.024012,30.4375,30.3750,0.0625,0.0000,30.386259,184664.953125,60.311018,...,2.9999,0.749700,58.475958,683887.530370,122.069181,1157.265457,1.367706e+08,1.0,0,101
1,30.41750,30.4375,0.029155,30.4375,30.3750,0.0625,0.0625,30.417514,185045.031250,60.378105,...,3.4051,0.898225,58.822494,692017.165572,122.363053,1164.164913,1.383955e+08,1.0,0,101
2,30.43750,30.4375,0.000000,30.4375,30.4375,0.0000,0.0000,30.437500,185288.281250,60.272277,...,2.8113,0.843775,59.034410,697012.314024,122.967085,1168.350106,1.393965e+08,1.0,0,101
3,30.43750,30.4375,0.000000,30.4375,30.4375,0.0000,0.0000,30.437500,185288.281250,60.272277,...,2.7495,0.739700,59.174283,700319.156483,123.569979,1171.086312,1.400580e+08,1.0,0,101
4,30.44750,30.4375,0.022913,30.5000,30.4375,0.0625,0.0000,30.447509,185410.156250,60.430399,...,2.7474,0.736750,59.049964,697379.651399,122.672981,1168.696010,1.394708e+08,1.0,0,101


In [17]:
SAVE_PATH = "/lustre09/project/6081099/reem2005/DATASET/pamap2_features.csv"

features_df.to_csv(
    SAVE_PATH,
    index=False
)

print("Feature dataset saved successfully!")
print("Saved to:", SAVE_PATH)

Feature dataset saved successfully!
Saved to: /lustre09/project/6081099/reem2005/DATASET/pamap2_features.csv
